In [ ]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu từ file order_items.csv (là file gốc chứa thông tin chi tiết item của đơn hàng)
order_items_path = '/content/order_items.csv'
df_items = pd.read_csv(order_items_path)

print("Kích thước ban đầu của order_items:", df_items.shape)
print("Các cột hiện có:", df_items.columns.tolist())
print("Số lượng dòng thiếu ở từng cột:\n", df_items.isnull().sum())

# Kiểm tra trùng lặp trên cặp (order_id, product_id)
dup_mask = df_items.duplicated(subset=['order_id', 'product_id'], keep=False)
print(f"\nSố lượng dòng bị trùng lặp theo cặp (order_id, product_id): {dup_mask.sum()}")

In [ ]:
# 2. Xử lý trùng lặp theo cặp (order_id, product_id), giữ lại dòng đầy đủ dữ liệu nhất
df_items['non_null_count'] = df_items.notnull().sum(axis=1)
df_items_sorted = df_items.sort_values(by='non_null_count', ascending=False)

df_items_cleaned = df_items_sorted.drop_duplicates(subset=['order_id', 'product_id'], keep='first').copy()
df_items_cleaned = df_items_cleaned.drop(columns=['non_null_count'])

print("Kích thước sau khi xử lý trùng lặp (order_id, product_id):", df_items_cleaned.shape)

In [ ]:
# 3. Điền khuyết dữ liệu thiếu (Imputation) theo các phương pháp phổ biến
# Chúng ta kiểm tra xem có trường nào được yêu cầu trong schema bị thiếu không
schema_cols = ['order_id', 'product_id', 'promo_id', 'promo_id_2', 'quantity', 'unit_price', 'discount_amount']

# Đảm bảo các cột schema có mặt trong dataframe
for col in schema_cols:
    if col not in df_items_cleaned.columns:
        df_items_cleaned[col] = np.nan

# Điền dữ liệu thiếu cho các cột trong schema
for col in schema_cols:
    if df_items_cleaned[col].isnull().any():
        if pd.api.types.is_numeric_dtype(df_items_cleaned[col]):
            median_val = df_items_cleaned[col].median()
            # Trường hợp cột toàn NaN không tính được median, ta điền mặc định là 0
            if pd.isna(median_val):
                median_val = 0
            df_items_cleaned[col] = df_items_cleaned[col].fillna(median_val)
            print(f"- Điền khuyết cột số '{col}' bằng: {median_val}")
        else:
            mode_val = df_items_cleaned[col].mode()
            if not mode_val.empty:
                fill_val = mode_val[0]
            else:
                fill_val = "None" # Sử dụng "None" cho promo_id nếu hoàn toàn trống
            df_items_cleaned[col] = df_items_cleaned[col].fillna(fill_val)
            print(f"- Điền khuyết cột phân loại '{col}' bằng: {fill_val}")

In [ ]:
# 4. Định dạng và lọc chính xác các cột theo cấu trúc bảng ORDER_ITEM rồi xuất file mới
order_item_final = df_items_cleaned[schema_cols].copy()

output_file = '/content/order_item_new.csv'
order_item_final.to_csv(output_file, index=False)

print(f"Đã xuất file thành công theo schema ORDER_ITEM tại: {output_file}")
print("Kích thước file mới:", order_item_final.shape)
print("\nXem trước 5 dòng đầu tiên:")
display(order_item_final.head())